# Asistente Experto con Gemini, RAG y Agentes

### Proyecto Final — Módulo de IA Generativa
**Dominio: Finanzas cuantitativas y mercados financieros**

---

Este notebook es **el entregable completo del proyecto**: contiene el código, la documentación,
la justificación de todas las decisiones de diseño y los ejemplos de uso. No necesitas ningún
otro archivo para entenderlo ni para evaluarlo.

Implementa un **agente conversacional experto** que responde preguntas sobre finanzas
cuantitativas apoyándose en su **propia base de conocimiento vectorial**, construida a partir de
**5 documentos académicos reales** de MIT, Columbia University e IARE.

| Componente | Tecnología |
|---|---|
| LLM | Google Gemini (`gemini-2.5-flash`) |
| Embeddings | Google Gemini (`gemini-embedding-001`) |
| Base vectorial | ChromaDB (persistente, distancia coseno) |
| Orquestación del agente | LangGraph (`StateGraph`) sobre LangChain |
| Memoria | `MemorySaver` (checkpointer por `thread_id`) |
| Interfaz bonus | Streamlit (desplegada en Streamlit Community Cloud)-[Link app StreamLit](https://proyecto-rag-nihalhk.streamlit.app/) |

## Índice

| Sección | Contenido |
|---|---|
| **0** | Mapa de cumplimiento del enunciado |
| **1** | Descripción del dominio y de la base de conocimiento |
| **2** | Arquitectura del sistema |
| **3** | Requisitos, instalación y ejecución |
| **4** | Configuración global e inicialización de los modelos |
| **5** | Descarga de los 5 documentos reales |
| **6** | Carga de los documentos |
| **7** | Segmentación en fragmentos (*chunking*) |
| **8** | Base de conocimiento vectorial (ChromaDB + Gemini Embeddings) |
| **9** | Prueba de recuperación |
| **10** | System prompt: diseño y **justificación** |
| **11** | El agente en LangGraph (RAG + memoria) |
| **12** | Función de conversación |
| **13** | **6 ejemplos documentados** (incluye memoria y fuera de ámbito) |
| **14** | **Celda de chat interactiva** |
| **15** | Justificación de las decisiones técnicas |
| **16** | Bonus: interfaz web en Streamlit (desplegada) |
| **17** | Fuentes, licencia y límites |

---
## 0. Mapa de cumplimiento del enunciado

Correspondencia directa entre cada requisito del enunciado y el punto del notebook donde se resuelve.

| Requisito del enunciado | Dónde está | ✔ |
|---|---|:--:|
| Agente experto en un dominio definido por el alumno | §1 — finanzas cuantitativas | ✔ |
| Base vectorial con **mínimo 3 documentos / ~20 páginas** | §5 — **5 PDFs**- 160 páginas | ✔ |
| Indexación en **ChromaDB con Gemini Embeddings** | §8 | ✔ |
| Preprocesado: *chunking*, limpieza | §7 | ✔ |
| **Verificar la colección con consultas de prueba** antes de conectar el agente | §9 | ✔ |
| Respuesta a preguntas usando **RAG + Gemini como LLM** | §11 | ✔ |
| **System prompt personalizado y justificado** | §10 | ✔ |
| Agente en **LangGraph** que use la base vectorial **en cada consulta** | §11 | ✔ |
| **Memoria de conversación** entre turnos | §11 (`MemorySaver` + `thread_id`) | ✔ |
| **Al menos 1 ejemplo que demuestre que la memoria funciona** | §13 — ejemplos **4 y 6** | ✔ |
| **Celda interactiva** para conversar con el agente | §14 | ✔ |
| **Mínimo 5 preguntas de ejemplo documentadas** | §13 — hay **6** | ✔ |
| Documentación: dominio, instalación, justificación del prompt, requisitos | §1, §3, §10, §15 | ✔ |
| **Bonus**: interfaz Streamlit desplegada | §16 | ✔ |

### Identificación del alumno

In [11]:
ALUMNO_UUID = "2a2215f6-6b66-4761-986a-35d29de2be24"

print("Alumno (UUID):", ALUMNO_UUID)

Alumno (UUID): 2a2215f6-6b66-4761-986a-35d29de2be24


---
## 1. Descripción del dominio y de la base de conocimiento

El dominio elegido es **finanzas cuantitativas y mercados financieros**: un campo con una
terminología precisa, mucha fórmula y muchos conceptos que se confunden entre sí (VaR frente a
Expected Shortfall, duración frente a convexidad, call frente a put). Es exactamente el tipo de
dominio donde un LLM genérico *suena* convincente pero se equivoca en los detalles, y donde por
tanto **el RAG aporta valor real**: anclar cada respuesta en un texto académico concreto.

La base de conocimiento está formada por **5 documentos PDF reales, de acceso público y
gratuito**, procedentes de material docente universitario. Suman en torno a **300 páginas**, muy
por encima del mínimo exigido (3 documentos / ~20 páginas).

| Archivo local | Fuente | Contenido |
|---|---|---|
| `01_derivados_opciones.pdf` | IARE — *Lecture Notes on Financial Derivatives* | Forwards, futuros, opciones (call/put), valoración, estrategias |
| `02_teoria_carteras.pdf` | Columbia University, M. Haugh — *Mean-Variance Analysis & CAPM* | Markowitz, frontera eficiente, CAPM, ratio de Sharpe |
| `03_gestion_riesgo.pdf` | Columbia University, M. Haugh — *Quantitative Risk Management* | Value at Risk, Expected Shortfall, medidas coherentes |
| `04_renta_fija.pdf` | J. Wang — *Fixed Income Securities* (material del MIT 15.401) | Bonos, duración, convexidad, sensibilidad a los tipos |
| `05_procesos_estocasticos.pdf` | MIT OCW 15.433 — *Random Walk on Wall Street* | Paseo aleatorio, movimiento browniano, mercados eficientes |

Los cinco documentos **están en inglés**, mientras que el agente responde **en español**: es una
decisión deliberada, porque obliga al modelo a comprender y traducir el contenido recuperado en
lugar de limitarse a copiarlo, y demuestra que la recuperación semántica funciona a través del
idioma (las preguntas se hacen en español y aun así recuperan fragmentos correctos en inglés).

> **Los PDFs no se distribuyen con el proyecto: el notebook los descarga automáticamente** desde
> sus URLs oficiales en la primera ejecución (§5). Así se evita redistribuir material docente
> ajeno y se garantiza que siempre se usa la versión original.

---
## 2. Arquitectura del sistema

```
  ┌─────────────────────── FASE DE INDEXACIÓN (una sola vez) ───────────────────────┐
  │                                                                                  │
  │   5 PDFs reales  ──►  Chunking          ──►  Embeddings      ──►  ChromaDB       │
  │   (descarga auto)     (1200 car. / 200      (gemini-              (persistente,  │
  │                        de solapamiento)      embedding-001)        coseno)       │
  └──────────────────────────────────────────────────────────────────────┬───────────┘
                                                                         │
  ┌─────────────────────── FASE DE CONSULTA (cada pregunta) ─────────────┼───────────┐
  │                                                                      ▼           │
  │                        ┌──────────────── Grafo LangGraph ────────────────┐       │
  │                        │                                                 │       │
  │   Pregunta  ──────────►│  START ──► recuperar ──────────► generar ──► END│──►  Respuesta
  │                        │              │                      │           │       │
  │                        │       reformula la pregunta   system prompt +   │       │
  │                        │       + búsqueda semántica     historial +      │       │
  │                        │       (top-k = 4)              contexto → Gemini│       │
  │                        └────────────────────┬────────────────────────────┘       │
  │                                             │                                    │
  │                        MemorySaver (checkpointer) — estado por thread_id         │
  └──────────────────────────────────────────────────────────────────────────────────┘
```

El grafo tiene **dos nodos**:

- **`recuperar`** — Si hay conversación previa, **reformula** la pregunta a una consulta autónoma
  (para que las preguntas de seguimiento sigan siendo buscables) y recupera de ChromaDB los
  `TOP_K` fragmentos más similares mediante búsqueda semántica.
- **`generar`** — Construye el prompt final (system prompt + historial + contexto recuperado +
  pregunta) y genera la respuesta con Gemini.

La **memoria** se implementa compilando el grafo con un *checkpointer* (`MemorySaver`), que guarda
el estado de cada conversación identificada por un `thread_id`.

---
## 3. Requisitos, instalación y ejecución

**Requisitos**
- **Python 3.10 o superior** (probado con 3.12).
- **API key de Google Gemini** (gratuita), desde [Google AI Studio](https://aistudio.google.com/apikey).
- Conexión a internet **solo la primera vez** (para descargar los 5 PDFs).

**Dependencias** (`requirements.txt`)

```
langchain>=0.3.0                 langgraph>=0.2.40
langchain-google-genai>=4.0.0    chromadb>=0.5.0
langchain-community>=0.3.0       pypdf>=4.0.0
langchain-text-splitters>=0.3.0  python-dotenv>=1.0.0
langchain-chroma>=0.1.4          requests>=2.31.0
jupyter>=1.0.0                   ipykernel>=6.0.0
streamlit>=1.36.0                pysqlite3-binary; sys_platform == "linux"   # bonus
```

> `langchain-google-genai` 4.x utiliza el SDK unificado `google-genai`; de ahí el requisito ≥ 4.0.

**Instalación y ejecución**

```bash
cd proyecto_final_rag

python -m venv .venv
source .venv/bin/activate        # Windows:  .venv\Scripts\activate

pip install -r requirements.txt

cp .env.example .env             

jupyter notebook                
```

La **clave nunca se escribe en el código**: se carga desde `.env` con `python-dotenv` (§4), y el
archivo `.env` está excluido en `.gitignore`.

La primera ejecución descarga los PDFs, genera los embeddings y crea la base vectorial en
`chroma_db/`. Las siguientes reutilizan ambas cosas automáticamente: **no se vuelve a descargar
ni a indexar**.

### 3.1 Instalación de dependencias

Ejecuta esta celda **una sola vez** si no has instalado las dependencias desde `requirements.txt`.

In [12]:
# %pip install -q langchain langchain-google-genai langchain-chroma langchain-community \
#                 langchain-text-splitters langgraph chromadb pypdf python-dotenv requests

print("Si no has instalado las dependencias, descomenta la línea %pip de esta celda.")

Si no has instalado las dependencias, descomenta la línea %pip de esta celda.


---
## 4. Configuración global e inicialización de los modelos

Todos los parámetros del proyecto viven **en un único lugar**, para poder ajustarlos sin tocar el
resto del código. La clave de la API se carga desde `.env`.

In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv

# --- Cargar la API key desde el archivo .env (nunca escribas la clave en el código) ---
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError(
        "No se encontró la clave de Gemini."
    )
    
# La librería langchain-google-genai busca la variable de entorno GOOGLE_API_KEY.
os.environ["GOOGLE_API_KEY"] = API_KEY

# --- Parámetros del pipeline ---
CARPETA_DOCS    = Path("documents")            # carpeta con los documentos fuente (PDFs reales)
CHROMA_DIR      = Path("chroma_db")            # carpeta donde se persiste la base vectorial
COLLECTION_NAME = "finanzas_cuantitativas"     # nombre de la colección en ChromaDB

MODELO_LLM       = "gemini-2.5-flash"          # LLM estable con capa gratuita
MODELO_EMBEDDING = "gemini-embedding-001"      # modelo de embeddings (GA)
# Si 'gemini-embedding-001' diera error de modelo no encontrado, prueba con
# "models/gemini-embedding-001".

CHUNK_SIZE    = 1200    # tamaño de cada fragmento (en caracteres)
CHUNK_OVERLAP = 200     # solapamiento entre fragmentos consecutivos
TOP_K         = 4       # nº de fragmentos que se recuperan por consulta

FORCE_REBUILD = False   # ponlo en True para reconstruir la base vectorial desde cero

print("Configuración cargada correctamente.")
print(f"  LLM        : {MODELO_LLM}")
print(f"  Embeddings : {MODELO_EMBEDDING}")
print(f"  Documentos : {CARPETA_DOCS.resolve()}")

Configuración cargada correctamente.
  LLM        : gemini-2.5-flash
  Embeddings : gemini-embedding-001
  Documentos : C:\Users\nihal\OneDrive\Desktop\proyecto_final_rag\documents


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Modelo de embeddings: transforma texto en vectores para la búsqueda semántica.
embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)

# Modelo de lenguaje. temperature baja -> respuestas más deterministas y fieles al
# contexto, lo ideal para RAG. (Si cambias a un modelo Gemini 3.x, deja la temperatura
# por defecto: en esos modelos, valores  pueden degradar el rendimiento.)
llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0.2)

print("Modelos de Gemini inicializados.")bajos

c:\Users\nihal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelos de Gemini inicializados.


---
## 5. Descarga de la base de conocimiento (5 documentos académicos reales)

La base de conocimiento **no se distribuye como archivos binarios**: se descarga de sus fuentes
oficiales la primera vez que ejecutas esta celda. Si un archivo ya existe en `documents/`, no se
vuelve a descargar.

> Si tu red bloquea alguna descarga, la celda te muestra la URL para bajar el PDF a mano y
> guardarlo en `documents/` con el mismo nombre.

**Para ampliar el dominio**: basta con añadir tus propios `.pdf`, `.txt` o `.csv` a `documents/`,
poner `FORCE_REBUILD = True` en §4 y reejecutar.

In [15]:
import requests

DOCUMENTOS_REALES = {
    "01_derivados_opciones.pdf": (
        "https://www.iare.ac.in/sites/default/files/lecture_notes/IARE_FD_NOTES.pdf",
        "IARE — Lecture Notes on Financial Derivatives",
    ),
    "02_teoria_carteras.pdf": (
        "https://www.columbia.edu/~mh2078/FoundationsFE/MeanVariance-CAPM.pdf",
        "Columbia University (M. Haugh) — Mean-Variance Analysis & CAPM",
    ),
    "03_gestion_riesgo.pdf": (
        "http://www.columbia.edu/~mh2078/RiskMeasures.pdf",
        "Columbia University (M. Haugh) — Quantitative Risk Management",
    ),
    "04_renta_fija.pdf": (
        "https://www.its.caltech.edu/~rosentha/courses/BEM103/Readings/JWCh03.pdf",
        "MIT 15.401 (J. Wang) — Fixed Income Securities",
    ),
    "05_procesos_estocasticos.pdf": (
        "https://ocw.mit.edu/courses/15-433-investments-spring-2003/c5845cd981c2e63f7ff303c92c7d41be_154332random_walk.pdf",
        "MIT OCW 15.433 — Random Walk on Wall Street",
    ),
}

CARPETA_DOCS.mkdir(exist_ok=True)

for nombre_archivo, (url, fuente) in DOCUMENTOS_REALES.items():
    destino = CARPETA_DOCS / nombre_archivo
    if destino.exists():
        print(f"  [ya existe] {nombre_archivo}")
        continue
    try:
        resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        destino.write_bytes(resp.content)
        print(f"  [descargado] {nombre_archivo}  <-  {fuente}")
    except Exception as e:
        print(f"  [ERROR] No se pudo descargar {nombre_archivo}: {e}")
        print(f"           Descárgalo manualmente desde: {url}")
        print(f"           y guárdalo como: {destino.resolve()}")

print("\nDescarga completada. Contenido de documents/:")
for f in sorted(CARPETA_DOCS.glob("*.pdf")):
    print(f"  - {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

  [descargado] 01_derivados_opciones.pdf  <-  IARE — Lecture Notes on Financial Derivatives
  [descargado] 02_teoria_carteras.pdf  <-  Columbia University (M. Haugh) — Mean-Variance Analysis & CAPM
  [descargado] 03_gestion_riesgo.pdf  <-  Columbia University (M. Haugh) — Quantitative Risk Management
  [descargado] 04_renta_fija.pdf  <-  MIT 15.401 (J. Wang) — Fixed Income Securities
  [descargado] 05_procesos_estocasticos.pdf  <-  MIT OCW 15.433 — Random Walk on Wall Street

Descarga completada. Contenido de documents/:
  - 01_derivados_opciones.pdf  (1024 KB)
  - 02_teoria_carteras.pdf  (442 KB)
  - 03_gestion_riesgo.pdf  (176 KB)
  - 04_renta_fija.pdf  (380 KB)
  - 05_procesos_estocasticos.pdf  (235 KB)


---
## 6. Carga de los documentos

Se recorre `documents/` y se usa el *loader* adecuado según la extensión de cada archivo, de modo
que el pipeline admite `.pdf`, `.txt` y `.csv` **sin tocar el código**.

In [16]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader

def cargar_documentos(carpeta: Path):
    """Recorre la carpeta y carga cada archivo con el loader adecuado según su extensión."""
    documentos = []
    for ruta in sorted(carpeta.rglob("*")):
        if ruta.is_dir():
            continue
        ext = ruta.suffix.lower()
        try:
            if ext in {".md", ".txt"}:
                documentos.extend(TextLoader(str(ruta), encoding="utf-8").load())
            elif ext == ".pdf":
                documentos.extend(PyPDFLoader(str(ruta)).load())
            elif ext == ".csv":
                documentos.extend(CSVLoader(str(ruta)).load())
            else:
                continue  # se ignoran otras extensiones (p.ej. .gitkeep)
            print(f"  [ok] {ruta.name}")
        except Exception as e:
            print(f"  [x]  {ruta.name}  ->  {e}")
    return documentos

print("Cargando documentos...")
documentos = cargar_documentos(CARPETA_DOCS)
print(f"\nTotal de páginas/documentos cargados: {len(documentos)}")

Cargando documentos...


D:\pip-temp\ipykernel_35800\4194789809.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader


  [ok] 01_derivados_opciones.pdf
  [ok] 02_teoria_carteras.pdf
  [ok] 03_gestion_riesgo.pdf
  [ok] 04_renta_fija.pdf
  [ok] 05_procesos_estocasticos.pdf

Total de páginas/documentos cargados: 160


---
## 7. Segmentación en fragmentos (*chunking*)

Los documentos se parten en fragmentos pequeños. Esto es **esencial** en RAG: permite recuperar
solo los trozos relevantes para cada pregunta, en vez de documentos enteros (que no cabrían en el
contexto y además diluirían la señal con ruido).

El **solapamiento** de 200 caracteres evita que una idea quede cortada a la mitad entre dos
fragmentos y pierda el contexto necesario para entenderse.

Se usa `RecursiveCharacterTextSplitter`, que intenta cortar primero por párrafos, luego por líneas
y solo como último recurso a mitad de palabra: así los fragmentos respetan la estructura del texto.

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],  # intenta cortar por párrafos primero
)
fragmentos = splitter.split_documents(documentos)

print(f"Se generaron {len(fragmentos)} fragmentos a partir de {len(documentos)} páginas/documentos.")
print("\nEjemplo de fragmento:\n" + "-" * 60)
print(fragmentos[0].page_content[:400], "...")
print("-" * 60)
print("Metadatos:", fragmentos[0].metadata)

Se generaron 303 fragmentos a partir de 160 páginas/documentos.

Ejemplo de fragmento:
------------------------------------------------------------
LECTURE NOTES 
 
ON 
 
FINANCIAL DERIVATIVES 
 
MBA IV SEMESTER (IARE – R16) 
 
 
 
 
 
 
Prepared by 
 
Mr. M Ramesh 
Assistant Professor, Department of MBA 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
DEPARTMENT OF MASTER OF BUSINESS ADMINISTRATION 
INSTITUTE OF AERONAUTICAL ENGINEERING 
(Autonomous) 
Dundigal, Hyderabad – 500 043 ...
------------------------------------------------------------
Metadatos: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2018-03-09T11:27:40+05:30', 'author': 'cisco', 'moddate': '2018-03-09T11:27:40+05:30', 'source': 'documents\\01_derivados_opciones.pdf', 'total_pages': 86, 'page': 0, 'page_label': '1'}


---
## 8. Base de conocimiento vectorial (ChromaDB + Gemini Embeddings)

Se generan los *embeddings* de cada fragmento con Gemini y se almacenan en **ChromaDB**, que
persiste en disco (`chroma_db/`). Se usa **distancia coseno**, la más adecuada para embeddings de
texto normalizados.

Dos decisiones importantes en esta celda:

1. **Indexación idempotente** — Solo se indexa si la colección está vacía (o si
   `FORCE_REBUILD = True`). Al reejecutar el notebook no se regeneran los embeddings: se ahorran
   cientos de llamadas a la API y varios minutos.

2. **Indexación por lotes con reintento automático** — La capa gratuita de Gemini limita las
   peticiones por minuto. Indexar ~300 fragmentos de golpe agota la cuota y devuelve un error
   `429 RESOURCE_EXHAUSTED`. Por eso se envían en **lotes de 20**, con una pausa entre lotes, y si
   aun así se alcanza el límite, se **lee el tiempo de espera que sugiere la propia API** y se
   reintenta ese lote. La función `con_reintento()` que se define aquí se reutiliza en **todo** el
   notebook, para que ninguna consulta se rompa a mitad de una demo por un límite de cuota.

In [ ]:
import time
import re

from langchain_chroma import Chroma

# Si FORCE_REBUILD es True, borramos la base anterior para reconstruirla desde cero.
if FORCE_REBUILD and CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
    print("Base vectorial anterior eliminada (FORCE_REBUILD=True).")

# Creamos (o abrimos) la colección persistente.
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

# --- Indexación por lotes con reintento automático (respeta el límite de la capa gratuita) ---
# La capa gratuita de Gemini limita las peticiones de embeddings por minuto. Con muchos
# fragmentos (como los que generan los 5 PDFs), indexar todo de golpe puede superar
# ese límite (error 429 RESOURCE_EXHAUSTED). 

TAMANO_LOTE = 20          # nº de fragmentos que se envían a Gemini por petición
PAUSA_ENTRE_LOTES = 3     # segundos de espera entre lotes (evita acercarse al límite)
ESPERA_POR_DEFECTO = 60   # segundos a esperar si no se puede leer el tiempo sugerido por la API

def _tiempo_de_espera_sugerido(error: Exception) -> float:
    """Extrae el 'Please retry in N s' del mensaje de error de Gemini, si está presente."""
    m = re.search(r"retry in ([\d.]+)s", str(error))
    return float(m.group(1)) + 2 if m else ESPERA_POR_DEFECTO

def indexar_por_lotes(vectorstore, fragmentos, tamano_lote=TAMANO_LOTE):
    total = len(fragmentos)
    for inicio in range(0, total, tamano_lote):
        lote = fragmentos[inicio: inicio + tamano_lote]
        intentos = 0
        while True:
            try:
                vectorstore.add_documents(lote)
                print(f"  [ok] fragmentos {inicio + 1}-{min(inicio + tamano_lote, total)} de {total}")
                break
            except Exception as e:
                if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                    intentos += 1
                    espera = _tiempo_de_espera_sugerido(e)
                    print(f"  [limite alcanzado] esperando {espera:.0f}s antes de reintentar "
                          f"(intento {intentos})...")
                    time.sleep(espera)
                else:
                    raise  # error distinto a límite de cuota: lo propagamos
        time.sleep(PAUSA_ENTRE_LOTES)

# Indexamos solo si la colección está vacía.
n_existentes = vectorstore._collection.count()
if n_existentes == 0:
    print(f"La base vectorial está vacía. Generando embeddings e indexando "
          f"{len(fragmentos)} fragmentos en lotes de {TAMANO_LOTE}...\n")
    indexar_por_lotes(vectorstore, fragmentos)
    print(f"\nIndexación completada: {vectorstore._collection.count()} fragmentos.")
else:
    print(f"Base vectorial existente cargada: {n_existentes} fragmentos.")
    print("(Pon FORCE_REBUILD=True si cambiaste los documentos y quieres reindexar.)")

# --- Función de reintento reutilizable para CUALQUIER llamada que use embeddings ---
# La usaremos también en la prueba de recuperación, en los ejemplos y en el chat,
# para que nunca se rompan por el límite de peticiones por minuto de la capa gratuita.
def con_reintento(func, *args, max_intentos=5, **kwargs):
    """Ejecuta func(*args, **kwargs); si Gemini devuelve 429 (límite de cuota),
    espera el tiempo sugerido y reintenta, hasta max_intentos veces."""
    for intento in range(1, max_intentos + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if ("RESOURCE_EXHAUSTED" in str(e) or "429" in str(e)) and intento < max_intentos:
                espera = _tiempo_de_espera_sugerido(e)
                print(f"  [limite alcanzado] esperando {espera:.0f}s antes de reintentar "
                      f"(intento {intento}/{max_intentos})...")
                time.sleep(espera)
            else:
                raise

print("\nFunción con_reintento() lista para usar en el resto del notebook.")

Base vectorial existente cargada: 301 fragmentos.
(Pon FORCE_REBUILD=True si cambiaste los documentos y quieres reindexar.)

Función con_reintento() lista para usar en el resto del notebook.


---
## 9. Prueba de recuperación

El enunciado pide **verificar que la colección vectorial responde correctamente a consultas de
prueba antes de conectar el agente**. Eso es justo lo que hace esta celda: lanza una consulta,
muestra los fragmentos más similares y su **distancia** (menor = más parecido), y de qué documento
salen.

Es el punto de control que separa «el RAG no funciona» de «el prompt no funciona»: si aquí los
fragmentos ya son irrelevantes, el problema está en la indexación, no en el LLM.

In [19]:
consulta_prueba = "¿Cómo se interpreta el ratio de Sharpe?"
resultados = con_reintento(vectorstore.similarity_search_with_score, consulta_prueba, k=TOP_K)

print(f"Consulta de prueba: {consulta_prueba!r}\n")
for i, (doc, score) in enumerate(resultados, 1):
    fuente = Path(doc.metadata.get("source", "?")).name
    fragmento = doc.page_content[:200].replace("\n", " ")
    print(f"[{i}] Fuente: {fuente}  |  distancia: {score:.4f}")
    print(f"    {fragmento} ...\n")

Consulta de prueba: '¿Cómo se interpreta el ratio de Sharpe?'

[1] Fuente: 02_teoria_carteras.pdf  |  distancia: 0.3087
    portfolio’s volatility. TheSharpe optimal portfoliois the portfolio with maximum Sharpe ratio. It is straightforward to see in our mean-variance framework (with a risk-free security) that the tangency ...

[2] Fuente: 02_teoria_carteras.pdf  |  distancia: 0.3222
    include the risk-free asset is also clear from (5) where we see thatσ min is linear inp. Note that this result is a 1-fund theoremsince every investor will optimally choose to invest in a combination  ...

[3] Fuente: 05_procesos_estocasticos.pdf  |  distancia: 0.3622
    Sample Statistics mean µ= 1 N N i=1 ri (33) variance: σ2 = 1 N N i=1 (ri −µ) 2 (34) skewness (lack of symmetry): skew= 1 N ∑N i=1 (ri −µ) 3 √ σ3 (35) kurtosis (peakedness): kurt= 1 N ∑N i=1 (ri −µ)  ...

[4] Fuente: 02_teoria_carteras.pdf  |  distancia: 0.3693
    Mean-Variance Optimization and the CAPM 7 α= 0must equal the slope of

---
## 10. System prompt: diseño y justificación

El *system prompt* define **quién es** el agente, **cómo se comporta** y —lo más importante— **qué
hace cuando no sabe algo**. Estas son las seis decisiones de diseño y el porqué de cada una:

**1. Rol y tono didáctico.**
Se define al agente como «QuantAsistente», experto en finanzas cuantitativas, con un estilo
*claro, riguroso y didáctico*. En un asistente experto no basta con acertar: tiene que **explicar**,
porque el usuario está aprendiendo. Un tecnicismo sin desarrollar es una respuesta fallida aunque
sea correcta.

**2. Fidelidad estricta al contexto.**
Se le obliga a responder **exclusivamente** con el CONTEXTO recuperado, sin inventar datos, cifras
ni fórmulas. Es la **salvaguarda principal contra las alucinaciones**, que son el mayor riesgo de
un asistente de dominio: en finanzas, una fórmula inventada que *parece* plausible es más
peligrosa que no responder.

**3. Honestidad ante la falta de información.**
Si el contexto no cubre la pregunta, debe reconocerlo abiertamente en lugar de improvisar. Un
*«eso no está en mi base de conocimiento»* es preferible a una respuesta falsa pero convincente.
→ *Se demuestra en el **ejemplo 5** de §13.*

**4. Trazabilidad de las fuentes.**
Se le pide citar el documento del que procede la información: el contexto que se le pasa incluye
la etiqueta `[Fuente: archivo, pág. N]`. Esto convierte la respuesta en **verificable** — el
usuario puede ir al PDF y comprobarla — y es lo que distingue un sistema RAG serio de un chatbot.

**5. Límite ético.**
El agente **no predice precios ni da recomendaciones de compra/venta**, y deja claro que su
contenido es **formativo y no constituye asesoramiento financiero**. En el dominio financiero esto
no es un adorno: es una precaución imprescindible.

**6. Coherencia conversacional.**
Se le indica que tenga en cuenta el historial para responder preguntas de seguimiento de forma
coherente. Sin esto, la memoria del grafo existiría pero el modelo no la usaría.

In [20]:
SYSTEM_PROMPT = (
    "Eres «QuantAsistente», un asistente experto en finanzas cuantitativas y mercados "
    "financieros. Tu misión es responder preguntas basándote EXCLUSIVAMENTE en el "
    "CONTEXTO recuperado de una base de conocimiento especializada (documentos académicos "
    "de MIT, Columbia University y otras instituciones).\n\n"
    "REGLAS DE COMPORTAMIENTO:\n"
    "1. Responde siempre en español, con un tono claro, riguroso y didáctico, aunque el "
    "material fuente esté en inglés. Traduce y explica los conceptos como lo haría un buen "
    "profesor: preciso pero accesible.\n"
    "2. Usa ÚNICAMENTE la información del CONTEXTO proporcionado. No inventes datos, cifras "
    "ni fórmulas que no aparezcan en él.\n"
    "3. Si el CONTEXTO no contiene la información necesaria, dilo con honestidad: indica que "
    "ese tema no está cubierto en tu base de conocimiento, en lugar de improvisar.\n"
    "4. Cuando sea útil, menciona de qué documento procede la información (aparece en el "
    "contexto, junto a la etiqueta 'Fuente').\n"
    "5. No haces predicciones de precios ni recomendaciones de compra/venta. Tus "
    "explicaciones son de carácter formativo y NO constituyen asesoramiento financiero.\n"
    "6. Si una pregunta hace referencia a algo dicho antes, ten en cuenta el historial de "
    "la conversación para mantener la coherencia."
)

print("System prompt definido.")

System prompt definido.


---
## 11. El agente en LangGraph (RAG + memoria)

El **estado** (`EstadoChat`) es lo que fluye entre nodos. El campo `messages` va anotado con
`add_messages`, lo que hace que el historial **se acumule** en vez de sobrescribirse: ahí vive la
memoria.

**Nodos del grafo**

1. **`recuperar`** — Si hay conversación previa, **reformula** la pregunta a una consulta autónoma
   y luego recupera los `TOP_K = 4` fragmentos más relevantes de ChromaDB.

2. **`generar`** — Construye el prompt final (system prompt + historial + contexto + pregunta) y
   genera la respuesta con Gemini.

**Por qué la reformulación es imprescindible.**
Una pregunta de seguimiento como *«¿y sus limitaciones?»* es, como texto, **inbuscable**: no
contiene ni una sola palabra del tema. Si la enviásemos tal cual a la búsqueda semántica,
recuperaríamos fragmentos aleatorios. Reformulándola con el historial se convierte en *«¿cuáles son
las limitaciones del Value at Risk como medida de riesgo?»*, que sí recupera lo correcto. **Es la
pieza que hace que RAG y memoria funcionen juntos** y no cada uno por su lado.

**La memoria** se logra compilando el grafo con un `MemorySaver` (*checkpointer*): guarda el estado
de cada conversación identificada por un `thread_id`, de modo que el agente «recuerda» los turnos
anteriores.

In [21]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver  # si fallara la importación: InMemorySaver

# ---- 1) Estado que fluye por el grafo ----
class EstadoChat(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]  # historial (se acumula solo)
    context: str                # fragmentos recuperados para el turno actual
    consulta_recuperacion: str  # consulta (reformulada) usada para buscar

# ---- 2) Reformulación de preguntas de seguimiento ----
def reformular_consulta(pregunta: str, historial: list) -> str:
    """Convierte una pregunta dependiente del contexto ('¿y sus limitaciones?') en una
    consulta de búsqueda autónoma, usando el historial de la conversación."""
    texto_historial = "\n".join(
        f"{'Usuario' if isinstance(m, HumanMessage) else 'Asistente'}: {m.content}"
        for m in historial[-6:]  # últimas ~3 rondas
    )
    instruccion = (
        "Dada la conversación previa y una nueva pregunta, reescribe la nueva pregunta como "
        "una consulta de búsqueda AUTÓNOMA y completa, entendible por sí sola sin ver la "
        "conversación. Si ya es autónoma, devuélvela tal cual. Responde SOLO con la consulta.\n\n"
        f"CONVERSACIÓN:\n{texto_historial}\n\n"
        f"NUEVA PREGUNTA: {pregunta}\n\nCONSULTA AUTÓNOMA:"
    )
    return con_reintento(llm.invoke, instruccion).content.strip()

# ---- 3) Nodo de recuperación (RAG) ----
def recuperar(estado: EstadoChat) -> dict:
    mensajes = estado["messages"]
    pregunta = mensajes[-1].content
    historial = mensajes[:-1]

    # Con conversación previa reformulamos; si no, buscamos la pregunta tal cual.
    consulta = reformular_consulta(pregunta, historial) if historial else pregunta

    # con_reintento protege esta búsqueda del límite de peticiones por minuto (429).
    docs = con_reintento(vectorstore.similarity_search, consulta, k=TOP_K)
    contexto = "\n\n---\n\n".join(
        f"[Fuente: {Path(d.metadata.get('source', '?')).name}, pág. {d.metadata.get('page', '?')}]\n{d.page_content}"
        for d in docs
    )
    return {"context": contexto, "consulta_recuperacion": consulta}

# ---- 4) Nodo de generación ----
def generar(estado: EstadoChat) -> dict:
    pregunta = estado["messages"][-1].content
    contexto = estado["context"]
    historial = estado["messages"][:-1]  # turnos anteriores = memoria

    prompt_turno = (
        "Responde a la PREGUNTA usando exclusivamente el siguiente CONTEXTO.\n\n"
        f"CONTEXTO:\n{contexto}\n\n"
        f"PREGUNTA: {pregunta}"
    )
    mensajes = [SystemMessage(content=SYSTEM_PROMPT)] + historial + [HumanMessage(content=prompt_turno)]
    respuesta = con_reintento(llm.invoke, mensajes)
    return {"messages": [respuesta]}

# ---- 5) Construcción y compilación del grafo ----
grafo = StateGraph(EstadoChat)
grafo.add_node("recuperar", recuperar)
grafo.add_node("generar", generar)
grafo.add_edge(START, "recuperar")
grafo.add_edge("recuperar", "generar")
grafo.add_edge("generar", END)

memoria = MemorySaver()  # checkpointer en memoria -> da memoria de conversación
agente = grafo.compile(checkpointer=memoria)

print("Agente LangGraph compilado:  START -> recuperar -> generar -> END")

Agente LangGraph compilado:  START -> recuperar -> generar -> END


In [22]:
# Estructura del grafo en formato Mermaid (se renderiza en GitHub o en https://mermaid.live)
try:
    print(agente.get_graph().draw_mermaid())
except Exception as e:
    print("No se pudo dibujar el grafo:", e)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	recuperar(recuperar)
	generar(generar)
	__end__([<p>__end__</p>]):::last
	__start__ --> recuperar;
	recuperar --> generar;
	generar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---
## 12. Función de conversación

`preguntar()` envía una pregunta al agente y devuelve su respuesta. El parámetro `thread_id`
identifica la conversación: preguntas con el **mismo** `thread_id` comparten memoria; con
**distinto** `thread_id` son conversaciones independientes.

Con `ver_consulta=True` se imprime además la **consulta reformulada** que se ha usado para buscar,
lo que permite ver la memoria funcionando por dentro.

In [23]:
def preguntar(texto: str, thread_id: str = "sesion-demo", ver_consulta: bool = False) -> str:
    """Envía una pregunta al agente y devuelve la respuesta.
    Mismas preguntas con el mismo thread_id comparten memoria."""
    config = {"configurable": {"thread_id": thread_id}}
    resultado = agente.invoke({"messages": [HumanMessage(content=texto)]}, config=config)
    if ver_consulta:
        print(f"   [consulta de búsqueda usada -> {resultado['consulta_recuperacion']!r}]")
    return resultado["messages"][-1].content

---
## 13. Ejemplos documentados

**6 preguntas de ejemplo** (el mínimo exigido son 5), todas con el mismo `thread_id`
(`"ejemplos"`), de modo que el agente mantiene el contexto entre ellas.

Cubren deliberadamente **tres situaciones distintas**:

| Ejemplo | Qué demuestra |
|---|---|
| **1 – 3** | Preguntas directas sobre **tres documentos distintos** de la base (carteras, derivados, riesgo) |
| **4** | **MEMORIA** — la pregunta dice *«ese problema»* sin nombrar el VaR; el agente lo deduce del historial |
| **5** | **FUERA DE ÁMBITO** — el agente reconoce que no puede responder, en lugar de inventar |
| **6** | **MEMORIA + SÍNTESIS** — se refiere a *«los tres conceptos de antes»*, cruzando toda la conversación |

> Ejecuta las celdas **en orden**, para que la memoria se vaya construyendo.

In [24]:
# --- Ejemplo 1: pregunta directa (teoría de carteras) ---
p1 = "¿Qué es el ratio de Sharpe y cómo se interpreta?"
print("PREGUNTA:", p1, "\n")
print("RESPUESTA:\n", preguntar(p1, thread_id="ejemplos"))

PREGUNTA: ¿Qué es el ratio de Sharpe y cómo se interpreta? 

RESPUESTA:
 El Ratio de Sharpe de una cartera (o de un valor) se define como la **relación entre el exceso de rentabilidad esperada de la cartera y la volatilidad de la cartera** (Fuente: 02_teoria_carteras.pdf, pág. 2).

En términos de interpretación, el Ratio de Sharpe mide la rentabilidad ajustada al riesgo de una inversión. Un Ratio de Sharpe más alto indica que la cartera está generando una mayor rentabilidad por cada unidad de riesgo asumido.

El **portafolio óptimo de Sharpe** (Sharpe optimal portfolio) es aquel que presenta el Ratio de Sharpe máximo (Fuente: 02_teoria_carteras.pdf, pág. 2). Dentro del marco de media-varianza y con la inclusión de un activo libre de riesgo, el portafolio de tangencia es el portafolio óptimo de Sharpe (Fuente: 02_teoria_carteras.pdf, pág. 2).


In [25]:
# --- Ejemplo 2: pregunta directa (derivados) ---
p2 = "¿Qué diferencia hay entre una opción call y una put?"
print("PREGUNTA:", p2, "\n")
print("RESPUESTA:\n", preguntar(p2, thread_id="ejemplos"))

PREGUNTA: ¿Qué diferencia hay entre una opción call y una put? 

RESPUESTA:
 La diferencia principal entre una opción "call" y una opción "put" radica en el derecho que otorgan a su tenedor:

*   **Opción Call (o de Compra):**
    *   Otorga al tenedor el **derecho a comprar** un activo específico a un precio de ejercicio (strike price) en o antes de la fecha de vencimiento (expiration date) (Fuente: 01_derivados_opciones.pdf, pág. 52, 38, 59).
    *   A cambio de este derecho, se paga una prima por adelantado al vendedor (Fuente: 01_derivados_opciones.pdf, pág. 52).
    *   Las opciones call suelen aumentar su valor a medida que el valor del activo subyacente se incrementa (Fuente: 01_derivados_opciones.pdf, pág. 52).
    *   Se abrevian como 'C' en las cotizaciones en línea (Fuente: 01_derivados_opciones.pdf, pág. 52).

*   **Opción Put (o de Venta):**
    *   Otorga al tenedor el **derecho a vender** un activo específico a un precio de ejercicio (strike price) en cualquier momento e

In [26]:
# --- Ejemplo 3: pregunta directa (gestión de riesgo) ---
p3 = "¿Qué es el Value at Risk (VaR) y por qué se dice que no es una medida coherente de riesgo?"
print("PREGUNTA:", p3, "\n")
print("RESPUESTA:\n", preguntar(p3, thread_id="ejemplos"))

PREGUNTA: ¿Qué es el Value at Risk (VaR) y por qué se dice que no es una medida coherente de riesgo? 

RESPUESTA:
 El Value at Risk (VaR) es una medida de riesgo que se define de la siguiente manera:

**Definición de VaR:**
Dado un nivel de confianza fijo $\alpha \in (0,1)$, el VaR de la pérdida de una cartera ($L$) en el intervalo de confianza $\alpha$ se expresa como:
$VaR_{\alpha} := q_{\alpha}(L) = \text{inf}\{x \in \mathbb{R} : F_L(x) \geq \alpha\}$
donde $F_L(\cdot)$ es la Función de Distribución Acumulada (CDF) de la variable aleatoria de pérdida $L$ (Fuente: 03_gestion_riesgo.pdf, pág. 1).
En términos más sencillos, el VaR representa el umbral de pérdida que no se espera exceder con una cierta probabilidad (nivel de confianza) durante un período de tiempo determinado.

**Por qué no es una medida coherente de riesgo:**
El Value at Risk **no es una medida de riesgo coherente** porque **falla en satisfacer el axioma de subaditividad**. Esta es quizás la principal crítica que se le

In [27]:
# --- Ejemplo 4: DEMOSTRACIÓN DE MEMORIA ---
# La pregunta no nombra el VaR ("esa medida"): el agente debe deducir, por el historial,
# que nos referimos al VaR del ejemplo 3. Activamos ver_consulta para ver la reformulación.
p4 = "¿Y qué alternativa se propone para corregir ese problema?"
print("PREGUNTA:", p4, "\n")
print("RESPUESTA:\n", preguntar(p4, thread_id="ejemplos", ver_consulta=True))

PREGUNTA: ¿Y qué alternativa se propone para corregir ese problema? 

  [limite alcanzado] esperando 41s antes de reintentar (intento 1/5)...
   [consulta de búsqueda usada -> '¿Qué alternativa se propone para corregir el problema de que el Value at Risk (VaR) no es una medida coherente de riesgo debido a que no cumple el axioma de subaditividad?']
RESPUESTA:
 El contexto proporcionado indica que las críticas a los axiomas de subaditividad y homogeneidad positiva (que el VaR no satisface) han llevado al estudio de las **medidas de riesgo convexas** (Fuente: 03_gestion_riesgo.pdf, pág. 1).

Una medida de riesgo convexa se propone como una alternativa que satisface los mismos axiomas que una medida de riesgo coherente, con la diferencia clave de que los axiomas de subaditividad y homogeneidad positiva son reemplazados por el **axioma de convexidad** (Fuente: 03_gestion_riesgo.pdf, pág. 1).

El axioma de convexidad se define como:
Para $L_1, L_2$ en el conjunto de pérdidas $M$ y $\lambda 

In [28]:
# --- Ejemplo 5: pregunta FUERA del ámbito de la base de conocimiento ---
# El agente debe reconocer que no dispone de esa información (ni predice precios),
# en lugar de inventar una respuesta.
p5 = "¿Cuánto valdrá el Bitcoin la semana que viene?"
print("PREGUNTA:", p5, "\n")
print("RESPUESTA:\n", preguntar(p5, thread_id="ejemplos"))

PREGUNTA: ¿Cuánto valdrá el Bitcoin la semana que viene? 

RESPUESTA:
 Lamento no poder responder a tu pregunta. El contexto proporcionado no contiene ninguna información sobre Bitcoin ni sobre predicciones de su valor futuro. Mi base de conocimiento se limita a la información contenida en los documentos que me has proporcionado, y estos no cubren ese tema.

Además, como asistente experto en finanzas cuantitativas, no realizo predicciones de precios ni recomendaciones de inversión.


In [29]:
# --- Ejemplo 6: MEMORIA + síntesis (referencia a toda la conversación) ---
p6 = "De los tres conceptos que te he preguntado antes, ¿cuál está más relacionado con medir el riesgo de una cartera?"
print("PREGUNTA:", p6, "\n")
print("RESPUESTA:\n", preguntar(p6, thread_id="ejemplos", ver_consulta=True))

PREGUNTA: De los tres conceptos que te he preguntado antes, ¿cuál está más relacionado con medir el riesgo de una cartera? 

   [consulta de búsqueda usada -> 'De los siguientes conceptos discutidos previamente: Value at Risk (VaR), medidas de riesgo coherentes y medidas de riesgo convexas, ¿cuál de ellos está más directamente relacionado con la acción de medir el riesgo de una cartera?']
RESPUESTA:
 De los tres conceptos que hemos discutido, el que está más directamente relacionado con medir el riesgo de una cartera es el **Value at Risk (VaR)**.

El contexto proporcionado define explícitamente el VaR como una "medida de riesgo" y discute sus propiedades en relación con la "riesgosidad de una cartera" (Fuente: 03_gestion_riesgo.pdf, pág. 0, 1). De hecho, el documento entero del que se extrae esta información se titula "Risk Measures, Risk Aggregation and Capital Allocation" (Medidas de Riesgo, Agregación de Riesgos y Asignación de Capital), lo que subraya su enfoque en la medición del

---
## 14. Celda de chat interactiva

Celda de conversación libre con el agente: escribes una pregunta, pulsas Enter y obtienes la
respuesta **al momento**.

Todas las preguntas de una misma ejecución comparten `thread_id`, así que el agente **mantiene la
memoria durante toda la sesión de chat**: puedes hacer preguntas de seguimiento
(*«¿y sus limitaciones?»*, *«explícamelo más sencillo»*, *«¿cómo se relaciona con lo anterior?»*)
y las entenderá.

Escribe **`salir`** (o deja la línea vacía) para terminar.

In [30]:
# ═════════════════ CHAT INTERACTIVO CON EL AGENTE ═════════════════
# Ejecuta esta celda y conversa con QuantAsistente. Escribe 'salir' para terminar.

THREAD_CHAT = "chat-interactivo"   # cambia este id para empezar una conversación desde cero

print("QuantAsistente — chat interactivo")
print("Pregunta lo que quieras sobre finanzas cuantitativas.")
print("Escribe 'salir' (o deja la línea vacía) para terminar.\n")
print("=" * 74)

while True:
    try:
        pregunta = input("\nTú: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n\nConversación interrumpida.")
        break

    if pregunta.lower() in {"salir", "exit", "quit", ""}:
        print("\nConversación terminada. ¡Hasta luego!")
        break

    respuesta = preguntar(pregunta, thread_id=THREAD_CHAT)
    print(f"\nQuantAsistente: {respuesta}")
    print("-" * 74)

QuantAsistente — chat interactivo
Pregunta lo que quieras sobre finanzas cuantitativas.
Escribe 'salir' (o deja la línea vacía) para terminar.


QuantAsistente: La pregunta que has formulado está incompleta. Por favor, proporciona la pregunta completa para que pueda responderla utilizando el contexto proporcionado.
--------------------------------------------------------------------------

QuantAsistente: Hola. La pregunta que has formulado está incompleta o no se relaciona con el contenido de la base de conocimiento proporcionada. Por favor, formula una pregunta específica sobre finanzas cuantitativas o mercados financieros para que pueda responderla utilizando el contexto.
--------------------------------------------------------------------------

QuantAsistente: El **Ratio de Sharpe** de una cartera (o un valor) es la relación entre el exceso de rendimiento esperado de la cartera y la volatilidad de la cartera (Fuente: 02_teoria_carteras.pdf, pág. 2).

El **portafolio óptimo de Shar

---
## 15. Justificación de las decisiones técnicas

| Decisión | Por qué |
|---|---|
| **Grafo `StateGraph` explícito** en lugar de un agente ReAct prehecho | Un agente ReAct **podría decidir no recuperar** y responder de memoria paramétrica. Con un grafo `recuperar → generar` fijo, **se consulta la base de conocimiento en cada pregunta**, sin excepción — que es justo lo que pide el enunciado. Además es mucho más transparente de explicar y de depurar. |
| **Reformulación de la consulta** antes de buscar | Sin ella, *«¿y sus limitaciones?»* no recuperaría nada útil. Es lo que hace que **RAG y memoria funcionen juntos**. |
| **Distancia coseno** en ChromaDB | La métrica estándar para embeddings de texto: mide orientación semántica, no magnitud. |
| **`gemini-2.5-flash` con `temperature=0.2`** | Modelo estable con capa gratuita. Temperatura baja → respuestas deterministas y fieles al contexto, que es lo que interesa en RAG (no queremos creatividad, queremos fidelidad). *Con modelos Gemini 3.x conviene dejar la temperatura por defecto.* |
| **`gemini-embedding-001`** | Modelo de embeddings en disponibilidad general (GA). |
| **Chunks de 1200 caracteres / 200 de solapamiento** | Compromiso entre precisión de recuperación (chunks pequeños → más precisos) y contexto suficiente por fragmento (chunks grandes → más comprensibles). El solapamiento evita cortar ideas por la mitad. |
| **`TOP_K = 4`** | Suficiente para cubrir el tema desde varios ángulos sin saturar el prompt de ruido. |
| **Indexación idempotente + por lotes con reintento** | No repetir cientos de llamadas a la API en cada ejecución, y no morir por un `429` a mitad de una demo. |
| **Documentos descargados en tiempo de ejecución** en vez de incluidos en el repo | Evita redistribuir material docente ajeno (derechos de autor) y garantiza que se usa siempre la versión original de cada fuente. |
| **`MemorySaver` (RAM)** en vez de `SqliteSaver` | Suficiente para el alcance del proyecto. Para persistir la memoria entre sesiones basta con sustituirlo (ver §17). |

---
## 16. Bonus: interfaz web en Streamlit

La misma lógica del notebook (embeddings → ChromaDB → grafo LangGraph con memoria) está envuelta
en una app de Streamlit, `app.py`, con:

- Interfaz de **chat** con historial de conversación.
- **Rastro de fuentes**: cada respuesta muestra **de qué documento y de qué página** sale, y los
  fragmentos recuperados se pueden desplegar para verificarlos uno a uno.
- La **consulta reformulada** se muestra cuando difiere de la pregunta original — la memoria del
  agente, visible.
- Preguntas sugeridas, reinicio de conversación y aviso de «no es asesoramiento financiero».

### 🔗 App desplegada: https://proyecto-rag-nihalhk.streamlit.app/


**En local:**
```bash
pip install -r requirements.txt
streamlit run app.py
```

**Notas de despliegue en Streamlit Community Cloud**

1. Subir el repositorio a GitHub y conectarlo en [share.streamlit.io](https://share.streamlit.io),
   con `app.py` como archivo principal.
2. Añadir `GOOGLE_API_KEY` en los **secrets** de la app (nunca en el código).
3. **`pysqlite3-binary`** en `requirements.txt` + sustitución del módulo `sqlite3` al arrancar
   `app.py`: la imagen de Streamlit Cloud trae un `sqlite3` anterior al 3.35 que ChromaDB rechaza.
   **Sin este parche la app no arranca.**
4. **Versionar `chroma_db/` en el repositorio.** El sistema de archivos de Streamlit Cloud es
   efímero: si la base vectorial no viaja en el repo, la app tendría que descargar los 5 PDFs y
   regenerar los ~300 embeddings **en cada arranque en frío** — lento y suficiente para agotar la
   cuota gratuita de Gemini justo durante la demo. Con el índice ya construido, la app carga al
   instante y solo llama a la API para vectorizar cada pregunta.

La celda siguiente **regenera `app.py`** a partir del notebook (no hace falta ejecutarla para el
resto del proyecto: está aquí para que este notebook sea autosuficiente).

In [ ]:
%%writefile app.py
"""
app.py — Interfaz web (BONUS) del Asistente Experto con Gemini, RAG y Agentes.

Reutiliza el mismo pipeline del notebook (Gemini Embeddings -> ChromaDB -> grafo
LangGraph con memoria) y añade:
  · Reintento automático ante el límite de cuota de la capa gratuita (error 429).
  · Indexación por lotes si la base vectorial aún no existe.
  · Rastro de fuentes: cada respuesta muestra de qué documento y página proviene.

Ejecutar en local:
    streamlit run app.py

Despliegue en Streamlit Community Cloud: sube el repositorio y añade GOOGLE_API_KEY
en los "secrets" de la app (nunca en el código).
"""

# --- Parche de sqlite3 para Streamlit Cloud -----------------------------------
# ChromaDB exige sqlite3 >= 3.35, pero la imagen de Streamlit Community Cloud trae
# una versión más antigua y la app falla al arrancar con:
#   "RuntimeError: Your system has an unsupported version of sqlite3".
# La solución oficial es instalar 'pysqlite3-binary' (ya está en requirements.txt)
# y sustituir el módulo sqlite3 ANTES de importar chromadb. En local no hace nada.
try:
    __import__("pysqlite3")
    import sys
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass  # en local se usa el sqlite3 del sistema, que ya es lo bastante nuevo
# ------------------------------------------------------------------------------

import os
import re
import time
import uuid
from pathlib import Path
from typing import Annotated, TypedDict

import requests
import streamlit as st
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# ============================ Configuración ============================
CARPETA_DOCS = Path("documents")
CHROMA_DIR = Path("chroma_db")
COLLECTION_NAME = "finanzas_cuantitativas"

MODELO_LLM = "gemini-2.5-flash"
MODELO_EMBEDDING = "gemini-embedding-001"

CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 1200, 200, 4
TAMANO_LOTE, PAUSA_ENTRE_LOTES = 20, 3

# Documentos reales de la base de conocimiento (mismas fuentes que el notebook).
DOCUMENTOS_REALES = {
    "01_derivados_opciones.pdf": (
        "https://www.iare.ac.in/sites/default/files/lecture_notes/IARE_FD_NOTES.pdf",
        "Derivados y opciones",
        "IARE · Lecture Notes on Financial Derivatives",
    ),
    "02_teoria_carteras.pdf": (
        "https://www.columbia.edu/~mh2078/FoundationsFE/MeanVariance-CAPM.pdf",
        "Teoría de carteras y CAPM",
        "Columbia University · M. Haugh",
    ),
    "03_gestion_riesgo.pdf": (
        "http://www.columbia.edu/~mh2078/RiskMeasures.pdf",
        "Gestión del riesgo (VaR)",
        "Columbia University · M. Haugh",
    ),
    "04_renta_fija.pdf": (
        "https://www.its.caltech.edu/~rosentha/courses/BEM103/Readings/JWCh03.pdf",
        "Renta fija y duración",
        "MIT 15.401 · J. Wang",
    ),
    "05_procesos_estocasticos.pdf": (
        "https://ocw.mit.edu/courses/15-433-investments-spring-2003/c5845cd981c2e63f7ff303c92c7d41be_154332random_walk.pdf",
        "Procesos estocásticos",
        "MIT OCW 15.433",
    ),
}

PREGUNTAS_SUGERIDAS = [
    "¿Qué es el ratio de Sharpe y cómo se interpreta?",
    "¿Qué diferencia hay entre una opción call y una put?",
    "¿Por qué se dice que el VaR no es una medida coherente de riesgo?",
    "¿Cómo afecta la duración al precio de un bono?",
]

SYSTEM_PROMPT = (
    "Eres «QuantAsistente», un asistente experto en finanzas cuantitativas y mercados "
    "financieros. Tu misión es responder preguntas basándote EXCLUSIVAMENTE en el "
    "CONTEXTO recuperado de una base de conocimiento especializada (apuntes académicos "
    "de MIT, Columbia University y otras instituciones).\n\n"
    "REGLAS DE COMPORTAMIENTO:\n"
    "1. Responde siempre en español, con un tono claro, riguroso y didáctico, aunque el "
    "material fuente esté en inglés.\n"
    "2. Usa ÚNICAMENTE la información del CONTEXTO. No inventes datos, cifras ni fórmulas.\n"
    "3. Si el CONTEXTO no contiene la información necesaria, dilo con honestidad en lugar "
    "de improvisar.\n"
    "4. Cuando sea útil, menciona de qué documento procede la información.\n"
    "5. No haces predicciones de precios ni recomendaciones de compra/venta. Tus "
    "explicaciones son formativas y NO constituyen asesoramiento financiero.\n"
    "6. Ten en cuenta el historial de la conversación para mantener la coherencia."
)

# ------------- Clave de API: .env (local) o secrets (Streamlit Cloud) -------------
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
try:
    API_KEY = API_KEY or st.secrets["GOOGLE_API_KEY"]
except Exception:
    pass
if API_KEY:
    os.environ["GOOGLE_API_KEY"] = API_KEY


# ==================== Resiliencia frente al límite de cuota ====================
def _espera_sugerida(error: Exception, por_defecto: float = 60.0) -> float:
    """Lee el 'Please retry in N s' que devuelve Gemini al agotar la cuota."""
    m = re.search(r"retry in ([\d.]+)s", str(error))
    return float(m.group(1)) + 2 if m else por_defecto


def con_reintento(func, *args, max_intentos: int = 5, **kwargs):
    """Ejecuta func(); si Gemini responde 429 (cuota agotada), espera y reintenta.

    La capa gratuita limita las peticiones por minuto, así que sin esto la app se
    caería a mitad de una conversación.
    """
    for intento in range(1, max_intentos + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            agotada = "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e)
            if agotada and intento < max_intentos:
                time.sleep(_espera_sugerida(e))
            else:
                raise


# ============================ Base de conocimiento ============================
def descargar_documentos():
    """Descarga los PDFs a documents/ si aún no están en disco."""
    CARPETA_DOCS.mkdir(exist_ok=True)
    for nombre, (url, _, _) in DOCUMENTOS_REALES.items():
        destino = CARPETA_DOCS / nombre
        if destino.exists():
            continue
        try:
            resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            destino.write_bytes(resp.content)
        except Exception as e:
            st.warning(f"No se pudo descargar {nombre}: {e}")


def cargar_y_trocear():
    """Carga los documentos de documents/ y los parte en fragmentos."""
    documentos = []
    for ruta in sorted(CARPETA_DOCS.rglob("*")):
        if ruta.is_dir():
            continue
        ext = ruta.suffix.lower()
        if ext == ".pdf":
            documentos.extend(PyPDFLoader(str(ruta)).load())
        elif ext in {".md", ".txt"}:
            documentos.extend(TextLoader(str(ruta), encoding="utf-8").load())
        elif ext == ".csv":
            documentos.extend(CSVLoader(str(ruta)).load())

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(documentos)


@st.cache_resource(show_spinner=False)
def construir_agente():
    """Prepara modelos, base vectorial y grafo. Se ejecuta una sola vez por sesión."""
    embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)
    llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0.2)

    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR),
        collection_metadata={"hnsw:space": "cosine"},
    )

    # Solo indexamos si la colección está vacía (p. ej. en un despliegue nuevo).
    if vectorstore._collection.count() == 0:
        with st.status("Preparando la base de conocimiento…", expanded=True) as estado:
            estado.write("Descargando los documentos académicos…")
            descargar_documentos()

            estado.write("Segmentando el texto en fragmentos…")
            fragmentos = cargar_y_trocear()

            barra = st.progress(0.0, text=f"Indexando 0 / {len(fragmentos)} fragmentos")
            for i in range(0, len(fragmentos), TAMANO_LOTE):
                lote = fragmentos[i: i + TAMANO_LOTE]
                con_reintento(vectorstore.add_documents, lote)
                hechos = min(i + TAMANO_LOTE, len(fragmentos))
                barra.progress(hechos / len(fragmentos),
                               text=f"Indexando {hechos} / {len(fragmentos)} fragmentos")
                time.sleep(PAUSA_ENTRE_LOTES)
            estado.update(label="Base de conocimiento lista", state="complete", expanded=False)

    # ---------------------------- Grafo LangGraph ----------------------------
    class EstadoChat(TypedDict):
        messages: Annotated[list[AnyMessage], add_messages]
        context: str
        consulta: str
        fuentes: list

    def reformular(pregunta: str, historial: list) -> str:
        """Convierte una pregunta de seguimiento en una consulta autónoma y buscable."""
        texto = "\n".join(
            f"{'Usuario' if isinstance(m, HumanMessage) else 'Asistente'}: {m.content}"
            for m in historial[-6:]
        )
        instruccion = (
            "Dada la conversación previa y una nueva pregunta, reescribe la nueva pregunta "
            "como una consulta de búsqueda AUTÓNOMA y completa, entendible por sí sola. Si "
            "ya es autónoma, devuélvela tal cual. Responde SOLO con la consulta.\n\n"
            f"CONVERSACIÓN:\n{texto}\n\nNUEVA PREGUNTA: {pregunta}\n\nCONSULTA AUTÓNOMA:"
        )
        return con_reintento(llm.invoke, instruccion).content.strip()

    def recuperar(estado: EstadoChat) -> dict:
        mensajes = estado["messages"]
        pregunta = mensajes[-1].content
        historial = mensajes[:-1]

        consulta = reformular(pregunta, historial) if historial else pregunta
        docs = con_reintento(vectorstore.similarity_search, consulta, k=TOP_K)

        contexto = "\n\n---\n\n".join(
            f"[Fuente: {Path(d.metadata.get('source', '?')).name}, "
            f"pág. {d.metadata.get('page', '?')}]\n{d.page_content}"
            for d in docs
        )
        fuentes = [
            {
                "archivo": Path(d.metadata.get("source", "?")).name,
                "pagina": d.metadata.get("page", "?"),
                "extracto": d.page_content[:320].replace("\n", " ").strip(),
            }
            for d in docs
        ]
        return {"context": contexto, "consulta": consulta, "fuentes": fuentes}

    def generar(estado: EstadoChat) -> dict:
        pregunta = estado["messages"][-1].content
        historial = estado["messages"][:-1]
        prompt_turno = (
            "Responde a la PREGUNTA usando exclusivamente el siguiente CONTEXTO.\n\n"
            f"CONTEXTO:\n{estado['context']}\n\nPREGUNTA: {pregunta}"
        )
        mensajes = [SystemMessage(content=SYSTEM_PROMPT)] + historial + [
            HumanMessage(content=prompt_turno)
        ]
        return {"messages": [con_reintento(llm.invoke, mensajes)]}

    grafo = StateGraph(EstadoChat)
    grafo.add_node("recuperar", recuperar)
    grafo.add_node("generar", generar)
    grafo.add_edge(START, "recuperar")
    grafo.add_edge("recuperar", "generar")
    grafo.add_edge("generar", END)

    agente = grafo.compile(checkpointer=MemorySaver())
    return agente, vectorstore._collection.count()


# ================================ Estilos ================================
ESTILOS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Spectral:ital,wght@0,300;0,500;1,300&family=Inter:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500&display=swap');

:root {
  --tinta:        #131A2B;
  --tinta-alta:   #1B2438;
  --regla:        #2E3A57;
  --pergamino:    #E9E3D6;
  --pergamino-2:  #96A0B4;
  --laton:        #C9A227;
}

.stApp { background: var(--tinta); }
#MainMenu, footer, header[data-testid="stHeader"] { visibility: hidden; }
.block-container { padding-top: 2.2rem; max-width: 820px; }

html, body, [class*="css"] { font-family: 'Inter', system-ui, sans-serif; }

/* ---------- Cabecera ---------- */
.cabecera { margin-bottom: 1.6rem; }
.eyebrow {
  font-family: 'IBM Plex Mono', monospace;
  font-size: .68rem; letter-spacing: .22em; text-transform: uppercase;
  color: var(--laton); margin-bottom: .5rem;
}
.titulo {
  font-family: 'Spectral', Georgia, serif;
  font-weight: 300; font-size: 2.9rem; line-height: 1.05;
  color: var(--pergamino); letter-spacing: -.015em; margin: 0;
}
.subtitulo {
  font-size: .93rem; color: var(--pergamino-2);
  margin-top: .55rem; max-width: 52ch;
}
.regla-laton {
  height: 1px; margin: 1.3rem 0 0;
  background: linear-gradient(90deg, var(--laton) 0%, var(--regla) 45%, transparent 100%);
}

/* ---------- Rastro de fuentes (elemento firma) ---------- */
.rastro { margin: .55rem 0 .2rem; padding-left: .85rem; border-left: 2px solid var(--laton); }
.rastro-titulo {
  font-family: 'IBM Plex Mono', monospace;
  font-size: .66rem; letter-spacing: .16em; text-transform: uppercase;
  color: var(--laton); margin-bottom: .45rem;
}
.chip {
  display: inline-block; margin: 0 .35rem .35rem 0; padding: .2rem .55rem;
  font-family: 'IBM Plex Mono', monospace; font-size: .72rem;
  color: var(--pergamino); background: var(--tinta-alta);
  border: 1px solid var(--regla); border-radius: 3px;
}
.chip .pag { color: var(--pergamino-2); }
.reformulada {
  font-family: 'IBM Plex Mono', monospace; font-size: .72rem;
  color: var(--pergamino-2); margin-top: .4rem;
}
.reformulada b { color: var(--laton); font-weight: 500; }
.extracto {
  font-size: .82rem; color: var(--pergamino-2); line-height: 1.55;
  border-left: 1px solid var(--regla); padding-left: .8rem; margin-bottom: .9rem;
}
.extracto .ref {
  display: block; font-family: 'IBM Plex Mono', monospace;
  font-size: .68rem; color: var(--laton); margin-bottom: .25rem;
}

/* ---------- Barra lateral ---------- */
[data-testid="stSidebar"] { background: var(--tinta-alta); border-right: 1px solid var(--regla); }
.lat-titulo {
  font-family: 'IBM Plex Mono', monospace;
  font-size: .66rem; letter-spacing: .16em; text-transform: uppercase;
  color: var(--laton); margin: .2rem 0 .7rem;
}
.doc { margin-bottom: .75rem; }
.doc-nombre { font-size: .85rem; color: var(--pergamino); }
.doc-fuente {
  font-family: 'IBM Plex Mono', monospace;
  font-size: .68rem; color: var(--pergamino-2); margin-top: .1rem;
}
.metrica {
  font-family: 'Spectral', serif; font-size: 2rem; font-weight: 300;
  color: var(--laton); line-height: 1;
}
.metrica-pie {
  font-family: 'IBM Plex Mono', monospace; font-size: .66rem;
  letter-spacing: .12em; text-transform: uppercase; color: var(--pergamino-2);
}
.aviso {
  font-size: .74rem; color: var(--pergamino-2); line-height: 1.5;
  border-top: 1px solid var(--regla); padding-top: .8rem; margin-top: .3rem;
}

@media (prefers-reduced-motion: reduce) { * { animation: none !important; transition: none !important; } }
</style>
"""

# ================================ Interfaz ================================
st.set_page_config(page_title="QuantAsistente", page_icon="§", layout="centered")
st.markdown(ESTILOS, unsafe_allow_html=True)

st.markdown(
    """
    <div class="cabecera">
      <div class="eyebrow">RAG · Gemini · ChromaDB · LangGraph</div>
      <h1 class="titulo">QuantAsistente</h1>
      <div class="subtitulo">
        Responde sobre finanzas cuantitativas leyendo apuntes de MIT y Columbia.
        Cada respuesta indica el documento y la página de los que sale.
      </div>
      <div class="regla-laton"></div>
    </div>
    """,
    unsafe_allow_html=True,
)

if not API_KEY:
    st.error(
        "Falta la clave de Gemini. Añade GOOGLE_API_KEY a un archivo .env (en local) "
        "o a los secrets de la app (en Streamlit Cloud)."
    )
    st.stop()

agente, n_fragmentos = construir_agente()

# ---- Estado de la sesión ----
st.session_state.setdefault("thread_id", str(uuid.uuid4()))
st.session_state.setdefault("historial", [])
st.session_state.setdefault("pendiente", None)


def formatear_rastro(fuentes: list, consulta: str, pregunta: str) -> str:
    """Construye el rastro de fuentes que acompaña a cada respuesta."""
    vistas, chips = set(), []
    for f in fuentes:
        clave = (f["archivo"], f["pagina"])
        if clave in vistas:
            continue
        vistas.add(clave)
        titulo = DOCUMENTOS_REALES.get(f["archivo"], (None, f["archivo"], None))[1]
        chips.append(f'<span class="chip">{titulo} <span class="pag">· p. {f["pagina"]}</span></span>')

    html = ['<div class="rastro"><div class="rastro-titulo">Fuentes consultadas</div>']
    html.append("".join(chips))
    if consulta and consulta.strip().lower() != pregunta.strip().lower():
        html.append(f'<div class="reformulada"><b>Búsqueda reformulada →</b> {consulta}</div>')
    html.append("</div>")
    return "".join(html)


# ---- Barra lateral ----
with st.sidebar:
    st.markdown('<div class="lat-titulo">Base de conocimiento</div>', unsafe_allow_html=True)
    st.markdown(
        f'<div class="metrica">{n_fragmentos}</div>'
        f'<div class="metrica-pie">fragmentos indexados</div>',
        unsafe_allow_html=True,
    )
    st.markdown('<div style="height:1.1rem"></div>', unsafe_allow_html=True)

    for _, (_, titulo, fuente) in DOCUMENTOS_REALES.items():
        st.markdown(
            f'<div class="doc"><div class="doc-nombre">{titulo}</div>'
            f'<div class="doc-fuente">{fuente}</div></div>',
            unsafe_allow_html=True,
        )

    st.markdown('<div class="lat-titulo" style="margin-top:1.4rem">Prueba a preguntar</div>',
                unsafe_allow_html=True)
    for sugerencia in PREGUNTAS_SUGERIDAS:
        if st.button(sugerencia, use_container_width=True, key=f"sug_{sugerencia[:18]}"):
            st.session_state.pendiente = sugerencia

    st.markdown('<div style="height:.8rem"></div>', unsafe_allow_html=True)
    if st.button("Empezar de nuevo", use_container_width=True):
        st.session_state.thread_id = str(uuid.uuid4())
        st.session_state.historial = []
        st.rerun()

    st.markdown(
        '<div class="aviso">Contenido formativo. No es asesoramiento financiero.</div>',
        unsafe_allow_html=True,
    )

# ---- Conversación ----
for turno in st.session_state.historial:
    with st.chat_message(turno["rol"]):
        st.write(turno["texto"])
        if turno.get("rastro"):
            st.markdown(turno["rastro"], unsafe_allow_html=True)
            with st.expander("Ver los fragmentos recuperados"):
                for f in turno["fuentes"]:
                    st.markdown(
                        f'<div class="extracto"><span class="ref">{f["archivo"]} · pág. {f["pagina"]}'
                        f'</span>{f["extracto"]}…</div>',
                        unsafe_allow_html=True,
                    )

pregunta = st.chat_input("Pregunta sobre finanzas cuantitativas…") or st.session_state.pendiente
st.session_state.pendiente = None

if pregunta:
    with st.chat_message("user"):
        st.write(pregunta)
    st.session_state.historial.append({"rol": "user", "texto": pregunta})

    with st.chat_message("assistant"):
        with st.spinner("Buscando en los apuntes…"):
            config = {"configurable": {"thread_id": st.session_state.thread_id}}
            resultado = agente.invoke(
                {"messages": [HumanMessage(content=pregunta)]}, config=config
            )
            respuesta = resultado["messages"][-1].content
            fuentes = resultado.get("fuentes", [])
            consulta = resultado.get("consulta", "")

        st.write(respuesta)
        rastro = formatear_rastro(fuentes, consulta, pregunta)
        st.markdown(rastro, unsafe_allow_html=True)
        with st.expander("Ver los fragmentos recuperados"):
            for f in fuentes:
                st.markdown(
                    f'<div class="extracto"><span class="ref">{f["archivo"]} · pág. {f["pagina"]}'
                    f'</span>{f["extracto"]}…</div>',
                    unsafe_allow_html=True,
                )

    st.session_state.historial.append(
        {"rol": "assistant", "texto": respuesta, "rastro": rastro, "fuentes": fuentes}
    )

---
## 17. Fuentes, licencia, límites y próximos pasos

**Fuentes y licencia.**
Los 5 documentos son material docente de acceso público, propiedad de sus autores e instituciones
(MIT, Columbia University, IARE). Se usan aquí **exclusivamente con fines educativos**, tal como
fueron publicados por sus autores para libre consulta. Las URLs de descarga están en §5. La carpeta
`documents/` se distribuye **vacía** (ver `.gitignore`): cada usuario descarga los PDFs al ejecutar
el notebook.

**Límites conocidos.**

- **Memoria en RAM.** `MemorySaver` pierde la memoria al reiniciar el kernel. Para persistirla
  entre sesiones basta con sustituirlo:
  ```python
  from langgraph.checkpoint.sqlite import SqliteSaver
  memoria = SqliteSaver.from_conn_string("memoria.sqlite")
  ```
- **Capa gratuita de Gemini.** Hay un límite de peticiones por minuto. El notebook ya lo gestiona
  con `con_reintento()`, pero ante un `429` persistente conviene esperar un minuto.
- **El agente solo sabe lo que hay en sus 5 documentos.** No es un fallo: es el diseño. Si le
  preguntas por criptomonedas o por el precio de mañana, debe decir que no lo sabe (§13, ejemplo 5).
- **Contenido formativo.** Nada de lo que responde este agente constituye asesoramiento financiero.

**Próximos pasos.**

- **Ampliar el dominio**: añadir `.pdf`, `.txt` o `.csv` a `documents/`, poner `FORCE_REBUILD = True`
  en §4 y reejecutar.
- **Reranking**: recuperar `k = 10` y reordenar con un cross-encoder antes de pasar el contexto al
  LLM, para subir la precisión.
- **Evaluación**: montar un conjunto de preguntas con respuesta conocida y medir *retrieval hit
  rate* y fidelidad de la respuesta al contexto.